In [3]:
import requests

BASE = "http://localhost:8000"

# Empleados disponibles en la DB:
#   id=1  dni=444555687  Maria Gonzalez    activa
#   id=2  dni=444545545  Rocio Gonzalez    INACTIVA
#   id=3  dni=85895623   Rogelio Johann    activo

## Subtarea 2 — Autocomplete de clientes

In [4]:
# Crear cliente de prueba (si no existe)
r = requests.post(f"{BASE}/clientes/", json={"nombre": "Texcom", "direccion": "Av. Siempre Viva 123"})
print(r.status_code, r.json())

201 {'nombre': 'Texcom', 'direccion': 'Av. Siempre Viva 123', 'id': 1}


In [8]:
# Búsqueda parcial por nombre → devuelve id + nombre + direccion
r = requests.get(f"{BASE}/clientes/", params={"search": "tex"})
clientes = r.json()
print(r.status_code)
clientes

200


[{'nombre': 'Texcom', 'direccion': 'Av. Siempre Viva 123', 'id': 1}]

In [9]:
# Guardar el id del cliente para las siguientes celdas
CLIENTE_ID = clientes[0]["id"]
print("cliente_id:", CLIENTE_ID)

cliente_id: 1


## Subtarea 1 — Registro de entrada/salida

In [10]:
# Registrar ENTRADA exitosa
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada",
    "dni": "444555687",
    "cliente_id": CLIENTE_ID,
    "direccion": "Av. Corrientes 123"
})
print(r.status_code, r.json())  # 201 {success: true}

201 {'success': True, 'message': 'Asistencia registrada'}


In [11]:
# Registrar SALIDA exitosa
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "salida",
    "dni": "444555687",
    "cliente_id": CLIENTE_ID,
    "direccion": "Av. Corrientes 123"
})
print(r.status_code, r.json())  # 201 {success: true}

201 {'success': True, 'message': 'Asistencia registrada'}


## Subtarea 3 — Validaciones

In [12]:
# Error: empleado inexistente → 404
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "0000000", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})
print(r.status_code, r.json())

404 {'success': False, 'message': 'Empleado no encontrado'}


In [14]:
# Error: empleado INACTIVO → 409
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "444545545", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})
print(r.status_code, r.json())

409 {'success': False, 'message': 'El empleado está inactivo'}


In [15]:
# Error: cliente inexistente → 404
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "444555687", "cliente_id": 9999, "direccion": "Calle X"
})
print(r.status_code, r.json())

404 {'success': False, 'message': 'Cliente no encontrado'}


In [16]:
# Error: salida sin entrada previa (Rogelio no tiene registros) → 409
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "salida", "dni": "85895623", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})
print(r.status_code, r.json())

409 {'success': False, 'message': 'No hay una entrada activa para registrar la salida'}


In [17]:
# Preparar: registrar entrada de Rogelio
requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "85895623", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})

# Error: dos entradas consecutivas → 409
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "85895623", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})
print(r.status_code, r.json())

409 {'success': False, 'message': 'Registro duplicado: ya se registró este tipo en el último minuto'}


In [20]:
# Error: datos faltantes / DNI inválido (muy corto) → 422 estandarizado
r = requests.post(f"{BASE}/asistencias/", json={
    "tipo": "entrada", "dni": "123", "cliente_id": CLIENTE_ID, "direccion": "Calle X"
})
print(r.status_code, r.json())

422 {'success': False, 'message': 'DNI inválido: debe tener entre 7 y 9 dígitos'}


## Subtarea 4 — Historial con filtros y paginación

In [19]:
# Historial completo (ordenado por fecha desc, paginado)
r = requests.get(f"{BASE}/asistencias/")
print(r.status_code, "registros:", len(r.json()))
r.json()

200 registros: 3


[{'id': 3,
  'tipo': 'entrada',
  'dni': '85895623',
  'cliente': 'Texcom',
  'direccion': 'Calle X',
  'fecha_hora': '2026-06-10T16:06:49.296162'},
 {'id': 2,
  'tipo': 'salida',
  'dni': '444555687',
  'cliente': 'Texcom',
  'direccion': 'Av. Corrientes 123',
  'fecha_hora': '2026-06-10T16:04:53.655240'},
 {'id': 1,
  'tipo': 'entrada',
  'dni': '444555687',
  'cliente': 'Texcom',
  'direccion': 'Av. Corrientes 123',
  'fecha_hora': '2026-06-10T16:04:44.171681'}]

In [21]:
# Filtros combinados: por DNI + tipo + rango de fechas
from datetime import date
hoy = date.today().isoformat()

r = requests.get(f"{BASE}/asistencias/", params={
    "dni": "444555687",
    "tipo": "entrada",
    "desde": hoy,
    "hasta": hoy,
    "skip": 0,
    "limit": 10
})
print(r.status_code)
r.json()

200


[{'id': 1,
  'tipo': 'entrada',
  'dni': '444555687',
  'cliente': 'Texcom',
  'direccion': 'Av. Corrientes 123',
  'fecha_hora': '2026-06-10T16:04:44.171681'}]